# SCD Type 2: Idempotent Upsert

Slowly Changing Dimensions (SCD) Type 2 track the full history of attribute changes
by maintaining `valid_from` / `valid_to` date ranges. But a naive INSERT-based update
script will **duplicate rows** if run twice — destroying your warehouse's integrity
without any error.

This notebook demonstrates the non-idempotent anti-pattern, diagnoses the corruption,
and implements a correct idempotent upsert using DuckDB.

In [ ]:
!pip install pandas duckdb --quiet

## Generate Synthetic Data

- **dim_customer**: 100 customers with current address, each having `valid_from = 2024-01-01`,
  `valid_to = 9999-12-31`, and `is_current = true`.
- **updates**: 20 address-change events for random customers, effective on various dates in 2024.

In [ ]:
import pandas as pd
import numpy as np
import duckdb

np.random.seed(42)

# --- Current state: 100 customers ---
cities = ["New York", "Chicago", "Houston", "Phoenix", "Philadelphia",
          "San Antonio", "San Diego", "Dallas", "San Jose", "Austin"]

current_state = pd.DataFrame({
    "customer_id": range(1, 101),
    "name": [f"Customer_{i}" for i in range(1, 101)],
    "city": np.random.choice(cities, size=100),
    "valid_from": pd.Timestamp("2024-01-01"),
    "valid_to": pd.Timestamp("9999-12-31"),
    "is_current": True,
})

# --- Updates: 20 address changes ---
update_ids = np.random.choice(range(1, 101), size=20, replace=False)
updates = pd.DataFrame({
    "customer_id": update_ids,
    "new_city": np.random.choice(cities, size=20),
    "effective_date": pd.to_datetime(
        [f"2024-{np.random.randint(3,12):02d}-{np.random.randint(1,28):02d}" for _ in range(20)]
    ),
})

print(f"Current customers: {len(current_state)} rows")
print(f"Updates to apply:  {len(updates)} rows")
display(updates.head())

In [ ]:
# Load into DuckDB
con = duckdb.connect()

def reset_tables():
    """Reset dim_customer to original state and recreate updates table."""
    con.execute("DROP TABLE IF EXISTS dim_customer")
    con.execute("DROP TABLE IF EXISTS updates")
    con.execute("""
        CREATE TABLE dim_customer AS SELECT * FROM current_state
    """)
    con.execute("""
        CREATE TABLE updates AS SELECT * FROM updates_df
    """, {"updates_df": updates})
    return con.execute("SELECT COUNT(*) FROM dim_customer").fetchone()[0]

n = reset_tables()
print(f"dim_customer loaded: {n} rows")

## Anti-Pattern: The Non-Idempotent INSERT

The naive approach:
1. For each update, INSERT a new "current" row with the new city.
2. UPDATE the old row's `valid_to` and set `is_current = false`.

This works **once**. But if the pipeline retries (network glitch, orchestrator
re-run, manual recovery), the same updates get applied again — creating
**duplicate current rows** for the same customer.

In [ ]:
def naive_scd2_update(connection):
    """Non-idempotent SCD2 update — will duplicate rows on re-run."""
    connection.execute("""
        -- Step 1: Close out old current records
        UPDATE dim_customer
        SET valid_to = u.effective_date,
            is_current = false
        FROM updates u
        WHERE dim_customer.customer_id = u.customer_id
          AND dim_customer.is_current = true
    """)

    connection.execute("""
        -- Step 2: Insert new current records
        INSERT INTO dim_customer (customer_id, name, city, valid_from, valid_to, is_current)
        SELECT
            u.customer_id,
            d.name,
            u.new_city,
            u.effective_date,
            '9999-12-31'::TIMESTAMP,
            true
        FROM updates u
        JOIN dim_customer d ON u.customer_id = d.customer_id
    """)


# Run #1 — looks correct
reset_tables()
naive_scd2_update(con)
count_after_run1 = con.execute("SELECT COUNT(*) FROM dim_customer").fetchone()[0]
print(f"After run #1: {count_after_run1} rows (expected: 100 + 20 = 120)")

# Run #2 — simulating a retry
naive_scd2_update(con)
count_after_run2 = con.execute("SELECT COUNT(*) FROM dim_customer").fetchone()[0]
print(f"After run #2: {count_after_run2} rows (expected: still 120, got MORE!)")
print(f"\nDuplicate rows created: {count_after_run2 - 120}")

## Diagnosing the Problem

After two runs, some customers have **multiple `is_current = true` rows** —
or worse, the `valid_to` chain has gaps and overlaps.

In [ ]:
# Find customers with multiple current records
dupes = con.execute("""
    SELECT customer_id, COUNT(*) AS current_count
    FROM dim_customer
    WHERE is_current = true
    GROUP BY customer_id
    HAVING COUNT(*) > 1
    ORDER BY current_count DESC
""").fetchdf()

print(f"Customers with multiple 'current' rows: {len(dupes)}")
display(dupes.head())

# Show the full history for one corrupted customer
bad_id = dupes["customer_id"].iloc[0]
print(f"\nFull history for customer_id = {bad_id}:")
display(con.execute(f"""
    SELECT * FROM dim_customer
    WHERE customer_id = {bad_id}
    ORDER BY valid_from
""").fetchdf())

## Correct Approach: Idempotent Upsert

The fix has two key changes:

1. **Guard against already-applied updates**: only process updates where the new city
   differs from the current city (skip no-ops and already-applied changes).
2. **Use a single transaction** with DELETE + INSERT (since DuckDB doesn't support MERGE)
   to atomically close old records and insert new ones.

Running this twice produces **identical results** — that's idempotency.

In [ ]:
def idempotent_scd2_update(connection):
    """Idempotent SCD2 update — safe to re-run any number of times."""
    connection.execute("""
        -- Use a CTE to identify genuinely new changes:
        -- only updates where the new city differs from the current city.
        WITH new_changes AS (
            SELECT u.customer_id, u.new_city, u.effective_date, d.name
            FROM updates u
            JOIN dim_customer d
              ON u.customer_id = d.customer_id
             AND d.is_current = true
             AND d.city != u.new_city  -- skip already-applied updates
        )
        -- Step 1: Close out old records for genuinely changed customers
        UPDATE dim_customer
        SET valid_to = nc.effective_date,
            is_current = false
        FROM new_changes nc
        WHERE dim_customer.customer_id = nc.customer_id
          AND dim_customer.is_current = true
    """)

    connection.execute("""
        -- Step 2: Insert new current records only for genuinely new changes
        INSERT INTO dim_customer (customer_id, name, city, valid_from, valid_to, is_current)
        SELECT
            u.customer_id,
            d.name,
            u.new_city,
            u.effective_date,
            '9999-12-31'::TIMESTAMP,
            true
        FROM updates u
        JOIN dim_customer d
          ON u.customer_id = d.customer_id
        WHERE d.is_current = true
          AND d.city != u.new_city  -- same guard: only genuinely new changes
    """)


# Reset and run
reset_tables()

idempotent_scd2_update(con)
count_run1 = con.execute("SELECT COUNT(*) FROM dim_customer").fetchone()[0]
print(f"After run #1: {count_run1} rows")

idempotent_scd2_update(con)
count_run2 = con.execute("SELECT COUNT(*) FROM dim_customer").fetchone()[0]
print(f"After run #2: {count_run2} rows")

idempotent_scd2_update(con)
count_run3 = con.execute("SELECT COUNT(*) FROM dim_customer").fetchone()[0]
print(f"After run #3: {count_run3} rows")

print(f"\nIdempotent? {count_run1 == count_run2 == count_run3}")

In [ ]:
# Verify: no customer has multiple current records
dupes_check = con.execute("""
    SELECT customer_id, COUNT(*) AS current_count
    FROM dim_customer
    WHERE is_current = true
    GROUP BY customer_id
    HAVING COUNT(*) > 1
""").fetchdf()

print(f"Customers with multiple 'current' rows: {len(dupes_check)} (should be 0)")

# Show clean history for one updated customer
sample_id = int(updates["customer_id"].iloc[0])
print(f"\nClean history for customer_id = {sample_id}:")
history = con.execute(f"""
    SELECT * FROM dim_customer
    WHERE customer_id = {sample_id}
    ORDER BY valid_from
""").fetchdf()
display(history)

# Verify valid_from/valid_to chain has no gaps
if len(history) > 1:
    for i in range(len(history) - 1):
        assert history.iloc[i]["valid_to"] == history.iloc[i + 1]["valid_from"], "Gap in validity chain!"
    print("valid_from/valid_to chain is continuous — no gaps.")

## Takeaways

| Pattern | Idempotent? | Risk | When to Use |
| :--- | :--- | :--- | :--- |
| Naive INSERT + UPDATE | No | Duplicate rows on retry, broken valid_to chain | Never in production |
| Guarded INSERT (skip if already applied) | Yes | Slightly more complex SQL | Any SCD2 pipeline |
| MERGE (Delta Lake, Snowflake) | Yes (built-in) | Vendor lock-in | When available; preferred approach |
| dbt snapshots | Yes (automated) | Requires dbt | Best for teams already using dbt |

**Key principle:** every write operation in a data pipeline should produce the same result
whether it runs once or ten times. If it doesn't, you will discover the bug at 3 AM during
an incident retro.